# Frequency Analysis

Corpus frequency (Infini-gram) → trajectory classification (Spearman correlation).  
No model needed — this is corpus-level analysis.

In [2]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
print(PROJECT_ROOT)

/Users/trishasalas/Repos/Research/tmlr


In [ ]:
# Cell 2: Frequency analysis — corpus counts + trajectory correlation
import importlib
import src.frequency
importlib.reload(src.frequency)
from src.frequency import run_frequency_analysis

freq_results = run_frequency_analysis(PROJECT_ROOT)
freq_results['freq_df']

In [ ]:
# Cell 3: Inspect Spearman results
freq_results['correlation']

## Token Competition Trace (exploratory)

Per-prompt investigation: what tokens compete at the decision point?
Requires a loaded model.

In [ ]:
# Cell 4: Load model for token competition trace
model_name = "pythia-160m"
model = HookedTransformer.from_pretrained(model_name)

print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

In [ ]:
# Cell 5: Token competition trace — the mechanistic half of the frequency hypothesis.
# Trace every compound that has a trajectory class, so competition (a high-freq
# competitor at the decision point) can be compared against regress vs. climb.
from src.frequency import COMPOUNDS, token_competition_trace, save_competition_trace

TRACE_COMPOUNDS = [
    "skip_link", "keyboard_navigation",                     # peak_regress
    "focus_indicator", "semantic_html", "closed_captions",  # never_emerges
    "screen_reader", "alt_text",                            # monotonic_climb
    "color_contrast",                                       # mixed
]
prompt_by_name = {name: prompt for name, _w1, _w2, prompt in COMPOUNDS}

traces = {}
for name in TRACE_COMPOUNDS:
    comp = token_competition_trace(model, prompt_by_name[name], top_k=10)
    save_competition_trace(comp, PROJECT_ROOT, model_name, name)
    traces[name] = comp
    print(f"\n{name}  —  {prompt_by_name[name]!r}")
    for tok, prob in comp["final_top"][:5]:
        print(f"    {tok!r:>15}  {prob:.4f}")

# Inspect any single trace, e.g. traces["skip_link"]["traces"]
traces["skip_link"]["traces"]

### Delete Model & Clear Cache

In [ ]:
# Cell 7: Free memory
import gc
del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"Memory cleared — GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB allocated")
elif device == "mps":
    torch.mps.empty_cache()
    print("Memory cleared")